In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import pandas as pd


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("devicharith/language-translation-englishfrench")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/devicharith/language-translation-englishfrench/versions/2


In [3]:
data = pd.read_csv("/root/.cache/kagglehub/datasets/devicharith/language-translation-englishfrench/versions/2/eng_-french.csv",names=["English","French"])
english_sentences = data["English"].tolist()
french_sentences = data["French"].tolist()


In [4]:
print(data.head())

                   English                  French
0  English words/sentences  French words/sentences
1                      Hi.                  Salut!
2                     Run!                 Cours !
3                     Run!                Courez !
4                     Who?                   Qui ?


In [5]:
tokenizer_eng = Tokenizer()
tokenizer_eng.fit_on_texts(english_sentences)
eng_seq = tokenizer_eng.texts_to_sequences(english_sentences)

tokenizer_fr = Tokenizer()
tokenizer_fr.fit_on_texts(french_sentences)
fr_seq = tokenizer_fr.texts_to_sequences(french_sentences)

In [6]:
print(eng_seq[0:5])
print(fr_seq[0:5])

[[291, 634, 3272], [2818], [429], [429], [79]]
[[18607, 18608, 18609], [4241], [6947], [18610], [32]]


In [7]:
vocab_size_eng = len(tokenizer_eng.word_index) + 1
vocab_size_fr = len(tokenizer_fr.word_index) + 1
print(vocab_size_eng)
print(vocab_size_fr)

14532
30664


In [8]:
max_length = max(len(seq) for seq in eng_seq + fr_seq)
eng_seq_padded = pad_sequences(eng_seq, maxlen=max_length, padding='post')
fr_seq_padded = pad_sequences(fr_seq, maxlen=max_length, padding='post')

In [9]:
embedding_dim = 256 # Dimension of the embedding space
units = 512 # Number of units in the LSTM layers for both the encoder and decoder

In [10]:
encoder_inputs = Input(shape=(max_length,))
enc_emb = Embedding(input_dim=vocab_size_eng, output_dim=embedding_dim)(encoder_inputs)
encoder_lstm = LSTM(units, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c] # Stores the LSTM’s final hidden and cell states,
                                    #which will be used to initialize the decoder for generating the output sequence.

In [11]:
decoder_inputs = Input(shape=(max_length,))
dec_emb_layer = Embedding(input_dim=vocab_size_fr, output_dim=embedding_dim)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(units, return_sequences=True, return_state=True) # return_sequences: Whether to return the last output in the output sequence,
                                                                     #or the full sequence
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(vocab_size_fr, activation='softmax')
output = decoder_dense(decoder_outputs)

In [12]:
model = Model([encoder_inputs, decoder_inputs], output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [13]:
X_train, x_temp, y_train, y_temp = train_test_split(eng_seq_padded, fr_seq_padded, test_size=0.2)
X_val, X_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5)


In [14]:
model.fit([X_train, X_train], y_train, validation_data=([X_val, X_val], y_val), epochs=5, batch_size=128)

Epoch 1/5
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 597s 537ms/step - accuracy: 0.8781 - loss: 1.2231 - val_accuracy: 0.8987 - val_loss: 0.7098
Epoch 2/5
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 631s 548ms/step - accuracy: 0.9019 - loss: 0.6693 - val_accuracy: 0.9071 - val_loss: 0.6003
Epoch 3/5
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 624s 550ms/step - accuracy: 0.9092 - loss: 0.5616 - val_accuracy: 0.9124 - val_loss: 0.5311
Epoch 4/5
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 622s 550ms/step - accuracy: 0.9149 - loss: 0.4819 - val_accuracy: 0.9166 - val_loss: 0.4846
Epoch 5/5
1098/1098 ━━━━━━━━━━━━━━━━━━━━ 622s 550ms/step - accuracy: 0.9200 - loss: 0.4207 - val_accuracy: 0.9197 - val_loss: 0.4537


In [15]:
model.evaluate([X_test, X_test], y_test)

549/549 ━━━━━━━━━━━━━━━━━━━━ 38s 69ms/step - accuracy: 0.9210 - loss: 0.4490


[0.4531265199184418, 0.9202125072479248]

In [26]:
def translate_sentence(sentence):
    seq = tokenizer_eng.texts_to_sequences([sentence])
    padded = pad_sequences(seq, maxlen=max_length, padding='post')
    translated = np.argmax(model.predict([padded, padded]), axis=-1)

    translated_sentence = []
    for i in translated[0]:
        if i in tokenizer_fr.index_word:
            translated_sentence.append(tokenizer_fr.index_word[i])
        else:
            translated_sentence.append(' ')

    return ' '.join(translated_sentence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
Input: I am french.
Translated: je suis français                                                                                                        


In [27]:
input_sentence = "I am french."
translated_sentence = translate_sentence(input_sentence)
print(f"Input: {input_sentence}")
print(f"Translated: {translated_sentence}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
Input: I am french.
Translated: je suis français                                                                                                        


In [28]:
input_sentence = "I love you."
translated_sentence = translate_sentence(input_sentence)
print(f"Input: {input_sentence}")
print(f"Translated: {translated_sentence}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
Input: I love you.
Translated: je t'aime                                                                                                          


In [40]:
input_sentence = "we can go study."
translated_sentence = translate_sentence(input_sentence)
print(f"Input: {input_sentence}")
print(f"Translated: {translated_sentence}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
Input: we can go study.
Translated: nous pouvons y étudier                                                                                                      
